# Fine-tuning a transformer for media bias detection (BABE)

Part of **Headline Bias Detector**. This notebook fine-tunes a pretrained transformer
(RoBERTa) to classify news sentences as **Biased** or **Non-biased**, using the
expert-labelled **BABE** dataset (Spinde et al., Findings of EMNLP 2021).

**Evaluation protocol (honest, reproducible):**
- Duplicate sentences removed before splitting, so no sentence appears in both train and test.
- **5-fold stratified cross-validation**: every sentence is used for testing exactly once,
  and the model never sees its test fold during training.
- Inside each fold, 10% of the training data is held out as a validation set to pick the best epoch.
  The test fold is used only once, at the end.
- Results reported as mean ± standard deviation across the 5 folds.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.
Takes about 15-25 minutes with `roberta-base`.

In [ ]:
!pip install -q transformers scikit-learn pandas
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

## 1. Settings

In [ ]:
MODEL_NAME = "roberta-base"   # alternatives: "distilroberta-base" (faster), "roberta-large" (slower, may need smaller batch)
EPOCHS      = 4
LR          = 2e-5
BATCH_SIZE  = 16
MAX_LEN     = 128
N_FOLDS     = 5
SEED        = 42

## 2. Load BABE (downloaded directly from the authors' GitHub repository)

In [ ]:
import pandas as pd, numpy as np

URL = ("https://raw.githubusercontent.com/Media-Bias-Group/"
       "Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/main/data/final_labels_SG2.csv")
df = pd.read_csv(URL, sep=";", encoding="utf-8-sig")
df = df[df["label_bias"].isin(["Biased", "Non-biased"])]
before = len(df)
df["text"] = df["text"].astype(str).str.strip()
df = df.drop_duplicates(subset="text").reset_index(drop=True)   # prevents train/test leakage
texts = df["text"].tolist()
labels = (df["label_bias"] == "Biased").astype(int).to_numpy()

print(f"Sentences: {len(df)} (removed {before - len(df)} duplicates)")
print(df["label_bias"].value_counts())
print(f"Majority-class baseline accuracy: {max(labels.mean(), 1 - labels.mean()):.1%}")

## 3. Training and evaluation code

In [ ]:
import random, time
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_loader(idx, shuffle):
    enc = tokenizer([texts[i] for i in idx], truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels[idx]))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

@torch.no_grad()
def predict(model, loader):
    model.eval(); preds = []
    for ids, mask, _ in loader:
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        preds.extend(logits.argmax(-1).cpu().tolist())
    return np.array(preds)

def scores(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(y, p), "precision": pr, "recall": rc, "f1": f1}

def run_fold(train_idx, test_idx, fold):
    set_seed(SEED + fold)
    tr_idx, val_idx = train_test_split(train_idx, test_size=0.1, stratify=labels[train_idx],
                                       random_state=SEED + fold)
    tr_loader, val_loader, te_loader = make_loader(tr_idx, True), make_loader(val_idx, False), make_loader(test_idx, False)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(tr_loader) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)

    best_f1, best_state = -1, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for ids, mask, y in tr_loader:
            out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        val = scores(labels[val_idx], predict(model, val_loader))
        print(f"  fold {fold} epoch {epoch}: validation F1 {val['f1']:.3f}")
        if val["f1"] > best_f1:   # model selection uses validation data only
            best_f1 = val["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    p = predict(model, te_loader)          # test fold touched exactly once
    del model; torch.cuda.empty_cache()
    return scores(labels[test_idx], p), confusion_matrix(labels[test_idx], p, labels=[0, 1])

## 4. Run 5-fold cross-validation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results, total_cm = [], np.zeros((2, 2), dtype=int)
t0 = time.time()
for fold, (train_idx, test_idx) in enumerate(skf.split(texts, labels), 1):
    r, cm = run_fold(train_idx, test_idx, fold)
    fold_results.append(r); total_cm += cm
    print(f"FOLD {fold}: accuracy {r['accuracy']:.1%}  F1 {r['f1']:.1%}   ({(time.time()-t0)/60:.1f} min elapsed)\n")

## 5. Results

In [ ]:
res = pd.DataFrame(fold_results, index=[f"fold {i}" for i in range(1, N_FOLDS + 1)])
print(f"Model: {MODEL_NAME}   epochs: {EPOCHS}   lr: {LR}   sentences: {len(texts)}\n")
print((res * 100).round(1).to_string())
print("\nMEAN +/- STD over folds:")
for m in ["accuracy", "precision", "recall", "f1"]:
    print(f"  {m:<10} {res[m].mean():.1%} +/- {res[m].std():.1%}")

(tn, fp), (fn, tp) = total_cm
print("\nConfusion matrix (all folds combined, every sentence tested once):")
print(f"                      Pred Non-biased   Pred Biased")
print(f"  Actual Non-biased   {tn:>15}   {fp:>11}")
print(f"  Actual Biased       {fn:>15}   {tp:>11}")

summary = {"model": MODEL_NAME, "epochs": EPOCHS, "lr": LR, "n_sentences": len(texts), "folds": N_FOLDS,
           "per_fold": fold_results,
           "mean": {m: float(res[m].mean()) for m in res}, "std": {m: float(res[m].std()) for m in res},
           "confusion_matrix_total": total_cm.tolist()}
import json; json.dump(summary, open("finetune_results.json", "w"), indent=2, default=float)
print("\nSaved finetune_results.json (Files panel on the left -> right-click -> Download)")